# 4. Construção do dataset final
Usa a base limpa (status 200) para recuperar revisões válidas na base original e gerar um novo dataset.

<!-- mkdocs: Carrega os arquivos Excel de entrada para a analise. -->

In [38]:
import pandas as pd

df_original = pd.read_excel("MLCQCodeSmellSamples.xlsx")
df_200 = pd.read_excel("MLCQ_status_200.xlsx")

## 4.1 Padronização dos nomes das colunas
Normaliza os nomes de colunas para evitar inconsistências de leitura.

In [39]:
df_original.columns = df_original.columns.str.strip()
df_200.columns = df_200.columns.str.strip()

## 4.2 Padronização da coluna de link
Uniformiza o campo de link nas duas bases para permitir o cruzamento.

In [40]:
LINK_COL = "link"

df_original[LINK_COL] = df_original[LINK_COL].astype(str).str.strip()
df_200[LINK_COL] = df_200[LINK_COL].astype(str).str.strip()

## 4.3 Conjunto de links válidos
Cria a lista de links válidos a partir da base com status 200.

In [41]:
links_validos = set(df_200[LINK_COL].dropna().unique())
print(f"Quantidade de links válidos: {len(links_validos)}")

Quantidade de links válidos: 4364


## 4.4 Filtragem da base original
Mantém na base original apenas os registros com links válidos.

In [42]:
df_original_200 = df_original[df_original[LINK_COL].isin(links_validos)].copy()

df_original_200 = df_original_200.drop_duplicates(
    subset=["sample_id", "smell", "reviewer_id"]
)

print(df_original_200.shape)

(13442, 15)


## 4.5 Quantificação da base após filtragem

Apresenta o número total de avaliações individuais (linhas) e o número de itens únicos após agregação por `sample_id` e `smell`.

Essa distinção é importante, pois a base original contém múltiplas avaliações por instância de código, sendo necessário diferenciar entre volume de avaliações e quantidade de itens analisados.

In [43]:
print("Total de avaliações (linhas):", len(df_original_200))

total_itens = df_original_200[["sample_id", "smell"]].drop_duplicates().shape[0]
print("Total de itens únicos (sample_id + smell):", total_itens)

Total de avaliações (linhas): 13442
Total de itens únicos (sample_id + smell): 8711


### 4.6 Quantidade de code smells por sample

Calcula, para cada `sample_id`, a quantidade de tipos distintos de code smells avaliados.

In [44]:
smells_por_sample = df_original_200.groupby("sample_id")["smell"].nunique()
print(smells_por_sample.value_counts().sort_index())

smell
1      17
2    4347
Name: count, dtype: int64


### 4.7 Identificação de samples com apenas um code smell

Seleciona os `sample_id` que apresentam apenas um tipo de code smell associado.

Essa etapa permite identificar possíveis exceções na base, considerando que, teoricamente, cada instância deveria possuir até dois smells, de acordo com seu tipo (`class` ou `method`).

In [45]:
samples_1_smell = smells_por_sample[smells_por_sample == 1].index
df_original_200[df_original_200["sample_id"].isin(samples_1_smell)] \
    .sort_values(["sample_id", "smell"])

,id,reviewer_id,sample_id,smell,severity,review_timestamp,type,code_name,repository,commit_hash,path,start_line,end_line,link,is_from_industry_relevant_project
5797,6372,7,4162084,long method,none,2019-04-23 17:17:44.338396,function,org.apache.chemistry.opencmis.inmemory.storedo...,git@github.com:apache/chemistry-opencmis.git,ef8513d708e5e21710afe5cafb8b32a62a0ae532,/chemistry-opencmis-server/chemistry-opencmis-...,408,421,https://github.com/apache/chemistry-opencmis/b...,1.0
5841,6416,7,4368201,data class,none,2019-04-23 17:23:24.146104,class,org.apache.cxf.tools.corba.processors.idl.Type...,git@github.com:apache/cxf.git,6bf89e9c8804c8845ec4d38583dd33eea8256439,/tools/corba/src/main/java/org/apache/cxf/tool...,34,199,https://github.com/apache/cxf/blob/6bf89e9c880...,1.0
5686,6261,7,4591956,long method,none,2019-04-23 15:20:12.195909,function,org.apache.gora.accumulo.store.AccumuloStore#c...,git@github.com:apache/gora.git,e325402aaa84d4f1501eb00c2c3f5c15e32713ca,/gora-accumulo/src/main/java/org/apache/gora/a...,839,857,https://github.com/apache/gora/blob/e325402aaa...,1.0
5826,6401,7,4651628,long method,none,2019-04-23 17:21:39.323789,function,org.apache.flink.contrib.streaming.state.Rocks...,git@github.com:apache/flink.git,8068c8775ad067d75828e6360e7e0994348da9b9,/flink-state-backends/flink-statebackend-rocks...,383,392,https://github.com/apache/flink/blob/8068c8775...,1.0
3711,4271,7,5460082,long method,none,2019-04-10 09:51:34.282990,function,org.apache.pulsar.broker.admin.v2.Namespaces#s...,git@github.com:apache/pulsar.git,044daf8d61328265640a5b3e5008fc04fac73efa,/pulsar-broker/src/main/java/org/apache/pulsar...,375,383,https://github.com/apache/pulsar/blob/044daf8d...,1.0
5860,6435,7,5763217,long method,none,2019-04-23 17:25:31.359564,function,org.apache.syncope.client.console.topology.Top...,git@github.com:apache/syncope.git,114c412afbfba24ffb4fbc804e5308a823a16a78,/client/idm/console/src/main/java/org/apache/s...,321,339,https://github.com/apache/syncope/blob/114c412...,1.0
80,608,7,5904455,long method,none,2019-03-27 10:51:09.362144,function,org.apache.xml.utils.UnImplNode#hasAttribute S...,git@github.com:apache/xalan-j.git,cba6d7fe7e93defecb98d155e2a780f8a3f1fbaa,/src/org/apache/xml/utils/UnImplNode.java,323,329,https://github.com/apache/xalan-j/blob/cba6d7f...,0.0
83,611,7,5923532,data class,none,2019-03-27 10:51:37.685949,class,org.apache.zeppelin.interpreter.thrift.RemoteI...,git@github.com:apache/zeppelin.git,4219d552349f8f7f3e6de34505b8a8ae9835f98b,/zeppelin-interpreter/src/main/java/org/apache...,18484,18515,https://github.com/apache/zeppelin/blob/4219d5...,1.0
5836,6411,7,6481321,data class,none,2019-04-23 17:22:56.762421,class,org.eclipse.paho.client.mqttv3.MqttAsyncClient...,git@github.com:eclipse/paho.mqtt.java.git,5af7b53499e7dbe45b7227b3d41fc870089c0033,/org.eclipse.paho.client.mqttv3/src/main/java/...,1376,1419,https://github.com/eclipse/paho.mqtt.java/blob...,1.0
5863,6438,7,6773572,long method,none,2019-04-23 17:25:40.892889,function,com.facebook.ads.sdk.Page.APIRequestGetInstagr...,git@github.com:facebook/facebook-java-business...,561f1a75e1220b55a160a1b92b0187f72be9cd08,/src/main/java/com/facebook/ads/sdk/Page.java,16709,16722,https://github.com/facebook/facebook-java-busi...,1.0


### 4.8 Quantidade de revisores por sample

Apresenta a distribuição do número de revisores que avaliaram cada `sample_id`.


In [46]:
df_original_200.groupby("sample_id")["reviewer_id"].nunique().value_counts().sort_index()

reviewer_id
1    3249
2     185
3     643
4     244
5      43
Name: count, dtype: int64

### 4.9 Padronização dos dados e criação da variável binária

Realiza a padronização dos campos textuais (`severity`, `smell` e `type`) e cria a variável binária `tem_smell`, que indica a presença (1) ou ausência (0) de code smell com base na severidade informada.

In [47]:
df_original_200["severity"] = df_original_200["severity"].astype(str).str.strip().str.lower()
df_original_200["smell"] = df_original_200["smell"].astype(str).str.strip().str.lower()
df_original_200["type"] = df_original_200["type"].astype(str).str.strip().str.lower()

df_original_200["tem_smell"] = (df_original_200["severity"] != "none").astype(int)

### 4.10 Agregação das avaliações por instância e tipo de smell

Agrupa as avaliações por (`sample_id`, `smell`), consolidando:

- número total de avaliadores
- votos indicando presença de smell
- votos indicando ausência de smell

In [48]:
avaliacoes_por_sample_smell = (
    df_original_200.groupby(["sample_id", "smell"])
.agg(
        total_avaliacoes=("reviewer_id", "nunique"),
        votos_tem_smell=("tem_smell", "sum"),
        votos_nao_tem_smell=("tem_smell", lambda x: (x == 0).sum())
    )
    .reset_index()
)

avaliacoes_por_sample_smell.head(20)

,sample_id,smell,total_avaliacoes,votos_tem_smell,votos_nao_tem_smell
0,3698323,blob,1,0,1
1,3698323,data class,1,0,1
2,3698602,feature envy,1,0,1
3,3698602,long method,1,0,1
4,3698665,feature envy,1,0,1
5,3698665,long method,1,0,1
6,3698860,feature envy,1,0,1
7,3698860,long method,1,0,1
8,3699227,feature envy,1,0,1
9,3699227,long method,1,0,1


### 4.13 Cálculo da proporção de votos e classificação do consenso

Calcula a proporção de votos positivos (`perc_tem_smell`) e classifica cada instância conforme o padrão de concordância entre os revisores:

- unanimidade de presença
- unanimidade de ausência
- maioria
- empate

Essa classificação permite analisar o grau de consistência das avaliações humanas.

In [49]:
avaliacoes_por_sample_smell["perc_tem_smell"] = (
    avaliacoes_por_sample_smell["votos_tem_smell"] /
    avaliacoes_por_sample_smell["total_avaliacoes"]
)

def classificar_situacao(row):
    tem = row["votos_tem_smell"]
    nao = row["votos_nao_tem_smell"]
    
    if tem == row["total_avaliacoes"]:
        return "unanimidade_tem"
    elif nao == row["total_avaliacoes"]:
        return "unanimidade_nao_tem"
    elif tem == nao:
        return "empate"
    elif tem > nao:
        return "maioria_tem"
    else:
        return "maioria_nao_tem"

avaliacoes_por_sample_smell["situacao"] = avaliacoes_por_sample_smell.apply(classificar_situacao, axis=1)

avaliacoes_por_sample_smell = avaliacoes_por_sample_smell.sort_values(
    by=["smell", "situacao", "sample_id"]
)

avaliacoes_por_sample_smell.head(20)

,sample_id,smell,total_avaliacoes,votos_tem_smell,votos_nao_tem_smell,perc_tem_smell,situacao
138,3789205,blob,4,2,2,0.5,empate
154,3796929,blob,4,2,2,0.5,empate
262,3860739,blob,4,2,2,0.5,empate
386,3922790,blob,4,2,2,0.5,empate
472,3969522,blob,4,2,2,0.5,empate
474,3970015,blob,4,2,2,0.5,empate
1159,4329859,blob,4,2,2,0.5,empate
1396,4466087,blob,4,2,2,0.5,empate
1446,4493302,blob,4,2,2,0.5,empate
1653,4610445,blob,4,2,2,0.5,empate


### 4.14 Exportação da base consolidada

Salva a base agregada em formato Excel, contendo os resultados consolidados das avaliações por instância e tipo de code smell.

Essa base será utilizada nas etapas subsequentes de análise e construção do ground truth.null

In [50]:
avaliacoes_por_sample_smell.to_excel(
    "MLCQ_resumo_avaliacoes_samplePorSmell.xlsx",
    index=False
)